In [1]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import time
from tqdm import tqdm
import uproot

In [2]:
parent = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/conf14"
file_path = "/mu3e_sort_run0001000.root"
in_dir = parent + file_path
out_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast"

In [3]:
def is_valid_file(path):
    path = Path(path)
    return path.is_file() and path.stat().st_size > 0

## `old preprocessing - defunct`

In [5]:
def hits_by_event(events, hits, tracks):
    """group hits by their tracks, and then for tracks in an event, group them together"""
    ordered_events = []
    
    for i in tqdm(range(len(events["hit_mc_i"])), desc='grouping'):
        event = hits[events["hit_mc_i"][i]]
        if len(event) == 0:
            continue

        track_id_array = np.array(event["tid"])
        sorted_idx = np.argsort(track_id_array)
        event_sorted = event[sorted_idx]
        track_id_sorted = track_id_array[sorted_idx]

        change_indices = np.flatnonzero(np.diff(track_id_sorted)) + 1

        per_track_hits = np.split(event_sorted, change_indices)
        unique_tids = track_id_sorted[np.r_[0, change_indices]]

        hit_tracks = {tid: hits for tid, hits in zip(unique_tids, per_track_hits)}

        ordered_events.append(hit_tracks)
        
    return ordered_events

def sort_hits_by_event(in_dir: str):
    """preprocess ROOT files into parquet files"""
    root_path = Path(in_dir)
    tree = uproot.open(root_path)
    
    # unpacking relevant ROOT TTrees
    event_idx = tree["mu3e"].arrays('hit_mc_i', library='ak')
    hits = tree["mu3e_mchits"].arrays(library='ak')
    tracks = tree["mu3e_mc_tracks"].arrays(library='ak')
    
    order = np.argsort(tracks['tid'])
    ordered_tracks = tracks[order]
    
    ordered_events = hits_by_event(event_idx, hits, tracks)
    return ordered_events, ordered_tracks

def event_to_table(event, ordered_tracks):   
    hit_data = []
    #for track_id, hit_array in tqdm(event.items()):
    for track_id, hit_array in event.items():
        for hit in hit_array:
            hit_data.append({
                "truth_tID": int(track_id),
                "hitID": int(hit["hid"]),
                "x": float(hit["x"]),
                "y": float(hit["y"]),
                "z": float(hit["z"]),
                "time": float(hit["time"]),
                "edep": float(hit["edep"]),
                "px": float(hit["px"]),
                "py": float(hit["py"]),
                "pz": float(hit["pz"])
            })


    ordered_track_ids = ordered_tracks['tid']
    track_data = []
    #for track_id in tqdm(event.keys()):
    for track_id in event.keys():
        idx = np.searchsorted(ordered_track_ids, track_id)
        track = ordered_tracks[idx]

        track_data.append({
                "tid": int(track["tid"]),
                "pdg": int(track["pdg"]),
                "vx": float(track["vx"]),
                "vy": float(track["vy"]),
                "vz": float(track["vz"]),
                "vt": float(track["vt"]),
                "px": float(track["px"]),
            "py": float(track["py"]),
            "pz": float(track["pz"])
        })
        
    return hit_data, track_data

def save_to_parquet(i, hit_data, track_data, out_dir):
    pd.DataFrame(hit_data).to_parquet(f'{out_dir}/event_{i}_hits.parquet')
    pd.DataFrame(track_data).to_parquet(f'{out_dir}/event_{i}_tracks.parquet')
    
def preprocess(in_dir: str, file_name:str, out_dir: str):
    """preprocess ROOT files into parquet files"""
    root_path = Path(in_dir)
    tree = uproot.open(root_path)
    
    # unpacking relevant ROOT TTrees
    event_idx = tree["mu3e"].arrays('hit_mc_i', library='ak')
    hits = tree["mu3e_mchits"].arrays(library='ak')
    tracks = tree["mu3e_mc_tracks"].arrays(library='ak')
    
    # sorting tracks
    order = np.argsort(tracks['tid'])
    ordered_tracks = tracks[order]
    
    ordered_events = hits_by_event(event_idx, hits, tracks)
    
    for i, event in tqdm(enumerate(ordered_events), total=len(ordered_events), desc='events'):
        hit_data, track_data = event_to_table(event, ordered_tracks)
        #save_to_parquet(i, hit_data, track_data, out_dir)

## `individual event to parquet`

### `original`

In [63]:
def root_to_DataFrame(in_dir:str, out_dir:str, tracks_per_fake_event:30, event_limit:None):
    path_in = Path(in_dir)
    path_out = Path(out_dir)
    
    # 1 --- READ AND CHUNK TRACKS ---
    print('loading global hit and track data, compiling events...')
    
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "x", "y", "z", "time", "edep", "px", "py", "pz"]
    
    with uproot.open(path_in) as file:
        tracks_flat = file["mu3e_mc_tracks"].arrays(track_fields, library="ak")
        hits_flat = file["mu3e_mchits"].arrays(hit_fields, library="ak")
        
    global_track_df = ak.to_pandas(tracks_flat)
    
    # 2. --- APPLY EVENT LIMIT ---
    if event_limit:
        track_limit = event_limit * tracks_per_fake_event
        print(f"TEST : limiting to {event_limit} compiled events.")
        global_track_df = global_track_df.head(track_limit)
        
    # 3. --- CREATE EVENT IDS ---
    N_tracks = len(global_track_df)
    N_chunks = N_tracks // tracks_per_fake_event + 1
    
    chunk_ids = np.arange(N_chunks)
    fake_event_ids = np.repeat(chunk_ids, tracks_per_fake_event)[:N_tracks]
    
    global_track_df["eventID"] = fake_event_ids
    global_track_df = global_track_df.sort_values(by='eventID').reset_index(drop=True)
    
    global_hit_df = ak.to_pandas(hits_flat).rename(columns={"tid":"trackID", "hid":"hitID"})
    
    print("\n--- Done ---")
    
    return global_track_df, global_hit_df

def iterate_per_event(global_track_df, global_hit_df, out_dir):
    path_out = Path(out_dir)
    
    # 4. --- ITERATE AND SAVE PER EVENT ---
    
    # group track df by event
    grouped_tracks = global_track_df.groupby('eventID')
    # get list of unique event ids
    event_ids_to_process = global_track_df['eventID'].unique()
    
    for eventID in tqdm(event_ids_to_process, desc='saving parquet files per event'):
        
        # get tracks for event
        track_in_event_df = grouped_tracks.get_group(eventID)
        
        # identify track IDs for event
        event_tids = track_in_event_df['tid'].unique()
        
        # filter global hits table to find hits belonging to these tids 
        df_hits_event = global_hit_df[global_hit_df['trackID'].isin(event_tids)].copy()
        
        # add the 'event_id' column to the hits
        df_hits_event['eventID'] = eventID
        
        # save to parquet
        event_str = str(eventID).zfill(6) # e.g., '000010'
        
        df_hits_event.to_parquet(path_out / f'event_{event_str}_hits.parquet', index=False)
        track_in_event_df.to_parquet(path_out / f'event_{event_str}_tracks.parquet', index=False)
        
        
    print("\n--- Done ---")
    print(f"Result: {len(event_ids_to_process)} event pairs saved to {path_out}")

#### `testing root -> dataframe -> parquet`

In [ ]:
global_track_df, global_hit_df = root_to_DataFrame(in_dir, out_dir, 50, 50)

loading global hit and track data, compiling events...
TEST : limiting to 50 compiled events.

--- Done ---


In [ ]:
iterate_per_event(global_track_df, global_hit_df, out_dir)

saving parquet files per event: 100%|██████████| 50/50 [01:05<00:00,  1.31s/it]


--- Done ---
Result: 50 event pairs saved to /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test


### `real events`

In [16]:
def load_real_event_data(path_in, path_out):
    print('Loading global hit and track data, compiling events...')
    
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "pdg", "x", "y", "z", "time", "edep", "px", "py", "pz"]
    
    with uproot.open(path_in) as file:
        #tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="ak")
        tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="pd")
        #hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="ak")
        hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="pd")        
        hit_mapping = file["mu3e"]['hit_mc_i'].array(library='ak')

    #global_track_df = ak.to_pandas(tracks_flat).rename(columns={'tid': 'trackID'})
    global_track_df = tracks_flat.rename(columns={'tid': 'trackID'})
    #global_hit_df = ak.to_pandas(hits_flat).rename(columns={"tid": "trackID", "hid": "hitID"})
    global_hit_df = hits_flat.rename(columns={"tid": "trackID", "hid": "hitID"})
    
    return global_track_df, global_hit_df, hit_mapping

In [8]:
def root_to_parquet_real_events(in_dir: str, out_dir: str, event_limit: int=None):
    """
    Convert ROOT file to 2 parquet files (tracks and hits) with original eventID column.
    For hits without an associated event, assigns eventID : -1.
    """    
    path_in = Path(in_dir)
    path_out = Path(out_dir)
    path_out.mkdir(parents=True, exist_ok=True)
    
    global_track_df, global_hit_df, hit_mapping = load_real_event_data(path_in, path_out)
    
    print('Using original simulation event mapping...')        
    flat_indices = np.concatenate(hit_mapping)
    event_ids = np.array(np.concatenate(
        [np.full(len(arr), i) for i, arr in enumerate(hit_mapping)]
    ))

    global_hit_df['eventID'] = -1
    # map IDs to specific rows by looking at hits
    global_hit_df.iloc[flat_indices, global_hit_df.columns.get_loc("eventID")] = event_ids

    # map tracks to real events by looking at constituent hits
    tid_to_event = global_hit_df.set_index('trackID')['eventID']
    tid_to_event = tid_to_event[~tid_to_event.index.duplicated(keep='first')]
    global_track_df['eventID'] = global_track_df['trackID'].map(tid_to_event)
    

    # applying event limit
    if event_limit:
        print(f'Limiting to {event_limit} compiled events...')
        global_hit_df = global_hit_df[global_hit_df["eventID"] < event_limit]
        global_track_df = global_track_df[global_track_df["eventID"] < event_limit]
        
    # formatting dataframe    
    global_hit_df['eventID'] = global_hit_df['eventID'].fillna(-1).astype(int)
    global_track_df['eventID'] = global_track_df['eventID'].fillna(-1).astype(int)

    global_hit_df = global_hit_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    global_track_df = global_track_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    
    # Save to single parquet files
    print("Saving tracks to parquet...")
    global_track_df.to_parquet(path_out / "all_tracks.parquet", index=False)
    
    print("Saving hits to parquet...")
    global_hit_df.to_parquet(path_out / "all_hits.parquet", index=False)
    
    print("\n--- Done ---")
    print(f"Saved {len(global_track_df['eventID'].unique())} events to:")
    print(f"  - {path_out / 'all_tracks.parquet'}")
    print(f"  - {path_out / 'all_hits.parquet'}")
    
    return global_track_df, global_hit_df

##### `testing`

In [17]:
global_track_df, global_hit_df, hit_mapping = load_real_event_data(path_in=Path(in_dir), path_out=Path(out_dir))

Loading global hit and track data, compiling events...


In [20]:
global_hit_df.head()

,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz
0,13,0,90,-13,11.073321,-1.602243,-20.517843,209.755974,0.0,0.0,0.0,-0.0
1,40,0,90,-13,10.183298,-0.313401,-23.192420,204.460771,0.0,0.0,-0.0,0.0
2,47,0,90,-13,-6.240615,-3.555553,31.064675,181.334617,0.0,-0.0,-0.0,0.0
3,63,0,90,-13,3.569605,4.482184,-34.896518,83.160980,0.0,0.0,-0.0,0.0
4,74,0,90,-13,2.568113,-4.893667,-35.411128,79.894650,0.0,0.0,0.0,0.0


### `fake events`

In [9]:
def load_data(path_in, path_out):
    print('Loading global hit and track data, compiling events...')
    
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "pdg", "x", "y", "z", "time", "edep", "px", "py", "pz"]
    
    with uproot.open(path_in) as file:
        tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="ak")
        hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="ak")

    global_track_df = ak.to_pandas(tracks_flat).rename(columns={'tid': 'trackID'})
    global_hit_df = ak.to_pandas(hits_flat).rename(columns={"tid": "trackID", "hid": "hitID"})
    
    return global_track_df, global_hit_df

In [3]:
def root_to_parquet_simple(in_dir: str, out_dir: str, tracks_per_fake_event: int = 30, event_limit: int = None):
    """
    Convert ROOT file to 2 parquet files (tracks and hits) with eventID column.
    """
    path_in = Path(in_dir)
    path_out = Path(out_dir)
    path_out.mkdir(parents=True, exist_ok=True)
    
    global_track_df, global_hit_df = load_data(path_in, path_out)
        
    # Apply event limit
    if event_limit:
        track_limit = event_limit * tracks_per_fake_event
        print(f"TEST: limiting to {event_limit} compiled events.")
        global_track_df = global_track_df.head(track_limit)
        
    print(f'MODE: Creating fake events ({tracks_per_fake_event} tracks/event)...')

    N_tracks = len(global_track_df)
    N_chunks = N_tracks // tracks_per_fake_event + 1
    chunk_ids = np.arange(N_chunks)

    fake_event_ids = np.repeat(chunk_ids, tracks_per_fake_event)[:N_tracks]
    global_track_df['eventID'] = fake_event_ids

    # map hits to fake events
    print('Mapping hits to events')
    tid_to_event = global_track_df.set_index('trackID')['eventID']
    global_hit_df['eventID'] = global_hit_df['trackID'].map(tid_to_event)
    
    # drop assignable hits (NaN valued eventID)
    #global_hit_df = global_hit_df.dropna(subset=['eventID']).reset_index(drop=True)
    
    # Final Cleanup: No hits dropped, just filled and sorted
    global_hit_df['eventID'] = global_hit_df['eventID'].fillna(-1).astype(int)
    global_track_df['eventID'] = global_track_df['eventID'].fillna(-1).astype(int)

    global_hit_df = global_hit_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    global_track_df = global_track_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    
    # Save to single parquet files
    print("Saving tracks to parquet...")
    global_track_df.to_parquet(path_out / "all_tracks.parquet", index=False)
    
    print("Saving hits to parquet...")
    global_hit_df.to_parquet(path_out / "all_hits.parquet", index=False)
    
    print("\n--- Done ---")
    print(f"Saved {len(global_track_df['eventID'].unique())} events to:")
    print(f"  - {path_out / 'all_tracks.parquet'}")
    print(f"  - {path_out / 'all_hits.parquet'}")
    
    return global_track_df, global_hit_df

#### `testing faster root -> dataframe -> parquet`

In [6]:
global_track_df, global_hit_df = root_to_parquet_simple(in_dir, out_dir, tracks_per_fake_event=30, event_limit=None)

Loading global hit and track data, compiling events...
MODE: Creating fake events (30 tracks/event)...
Mapping hits to events
Saving tracks to parquet...
Saving hits to parquet...

--- Done ---
Saved 70466 events to:
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_tracks.parquet
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_hits.parquet


### `real events again but for real this time`

In [4]:
def load_real_event_data(path_in, path_out):
    print('Loading global hit and track data, compiling events...')
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "pdg", "x", "y", "z", "time", "edep", "px", "py", "pz"]
    
    with uproot.open(Path(in_dir)) as file:
        # pd.DataFrame of track and hit data from mu3e tree - only with chosen fields
        tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="pd")
        hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="pd")

        # jagged awkward array - each entry in jagged array = list of hit indices in mchits tree, each list = one event
        hit_mapping = file["mu3e"]["hit_mc_i"].array(library='ak') 

    # dataframe tidying    
    global_track_df = tracks_flat.rename(columns={'tid': 'trackID'})
    global_hit_df = hits_flat.rename(columns={"tid": "trackID", "hid": "hitID"})
        
    return global_track_df, global_hit_df, hit_mapping

In [5]:
def even_finaler_preprocessing(in_dir: str, out_dir: str, event_limit: int=None):
    """
    Convert ROOT file to 2 parquet files (tracks and hits) with original eventID column.
    For hits without an associated event, assigns eventID : -1.
    """   
    path_in = Path(in_dir)
    path_out = Path(out_dir)
    path_out.mkdir(parents=True, exist_ok=True)
    
    ### loading hit and track dataframes ###
    global_track_df, global_hit_df, hit_mapping = load_real_event_data(path_in, path_out)
    
    ### mapping events ###
    print('Mapping events...')
    # flattening into one long numpy array of hit indices - all the hits in mchits that are included in monte carlo eventing
    flat_indices = np.asarray(ak.flatten(hit_mapping), dtype=np.int64)

    # creates array of eventIDs - maps onto flat_indices --- first N entries in event_ids = 0, maps to first N hit indices in flat_indices
    event_ids = np.repeat(np.arange(len(hit_mapping)), ak.num(hit_mapping))

    # above misses out all events that aren't accounted for in the mu3e tree - below assigns those events eventID = -1 -- makes easy to filter out later
    full_event_ids = np.full(len(global_hit_df), -1, dtype=np.int32)   # array of -1, length = total number of hits
    full_event_ids[flat_indices] = event_ids                           # applies eventIDs to all hit idx
    global_hit_df['eventID'] = full_event_ids                          # applied eventIDs to all hits in global hit dataframe

    tid_to_event = global_hit_df[global_hit_df['eventID'] != -1].set_index('trackID')['eventID']
    tid_to_event = tid_to_event[~tid_to_event.index.duplicated(keep='first')]
    global_track_df['eventID'] = global_track_df['trackID'].map(tid_to_event).fillna(-1).astype(int)
    
    ### applying event limit ###
    if event_limit:
        print(f'Limiting to {event_limit} compiled events...')
        global_hit_df = global_hit_df[global_hit_df["eventID"] < event_limit]
        global_track_df = global_track_df[global_track_df["eventID"] < event_limit]
        print('Ordering dataframes...')
    else:
        print('Ordering dataframes...')
        
    # sorting eventID and trackID
    global_hit_df = global_hit_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    global_track_df = global_track_df.sort_values(['eventID', 'trackID']).reset_index(drop=True)
    
    # save to single parquet files
    print("Saving tracks to parquet...")
    global_track_df.to_parquet(path_out / "all_tracks.parquet", index=False)
    
    print("Saving hits to parquet...")
    global_hit_df.to_parquet(path_out / "all_hits.parquet", index=False)
    
    print("\n--- Done ---")
    print(f"Saved {len(global_track_df['eventID'].unique())} events to:")
    print(f"  - {path_out / 'all_tracks.parquet'}")
    print(f"  - {path_out / 'all_hits.parquet'}")
    
    return global_track_df, global_hit_df

#### `testing`

In [42]:
global_track_df, global_hit_df = even_finaler_preprocessing(in_dir, out_dir, event_limit=None)

Loading global hit and track data, compiling events...
Mapping events...
Ordering dataframes...
Saving tracks to parquet...
Saving hits to parquet...

--- Done ---
Saved 49974 events to:
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_tracks.parquet
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_hits.parquet


## `ensuring content of parquet files`

In [44]:
import pyarrow.parquet as pq

def read_parquet_schema_df(uri: str) -> pd.DataFrame:
    """Return a Pandas dataframe corresponding to the schema of a local URI of a parquet file.

    The returned dataframe has the columns: column, pa_dtype
    """
    # Ref: https://stackoverflow.com/a/64288036/
    schema = pq.read_schema(uri, memory_map=True)
    schema = pd.DataFrame(({"column": name, "pa_dtype": str(pa_dtype)} for name, pa_dtype in zip(schema.names, schema.types)))
    schema = schema.reindex(columns=["column", "pa_dtype"], fill_value=pd.NA)  # Ensures columns in case the parquet file has an empty dataframe.
    return schema

In [9]:
hit_file = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test/event_000000_hits.parquet'
track_file = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test/event_000000_tracks.parquet'

hit_file_path = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_hits.parquet'
track_file_path = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_tracks.parquet'

In [48]:
print(read_parquet_schema_df(track_file_path))

    column pa_dtype
0  trackID    int64
1      pdg    int32
2       vx   double
3       vy   double
4       vz   double
5       vt   double
6       px   double
7       py   double
8       pz   double
9  eventID    int64


In [29]:
NaNeventID_df = global_hit_df.sort_values('eventID').tail(7994)
len(NaNeventID_df[NaNeventID_df['pdg'] == -13])

7994

In [21]:
# list of event names for mu3e parquet files
track_file_path = Path(track_file_path)
event_id_series = pq.read_table(track_file_path, columns=['eventID']).to_pandas()['eventID']
# get unique and sorted eventIDs
unique_event_ids = event_id_series.unique()
unique_event_ids.sort()
# create standarised event names and sample ids
event_names = [f'event{ID:09d}' for ID in unique_event_ids]
sample_ids = unique_event_ids.tolist()

In [22]:
idx = 1

sample_id = sample_ids[idx]
event_name = event_names[idx]

all_hits = pd.read_parquet(hit_file_path)
all_tracks = pd.read_parquet(track_file_path)

hits = all_hits[all_hits['eventID']==sample_id]
tracks = all_tracks[all_tracks['eventID']==sample_id]

In [23]:
hits.sort_values('trackID')

,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz,eventID
113,63,0,90,-13,3.569605,4.482184,-34.896518,83.160980,0.000000,0.000000,-0.000000,0.000000,1.0
91,69,3,30,-11,-32.118667,50.057864,-312.470000,639.888732,0.571300,-1.099446,-14.990384,-15.397888,1.0
93,69,1,20,-11,60.364891,9.409642,-96.057101,638.833407,0.133549,13.478306,10.149418,-16.696679,1.0
99,69,2,10,-11,70.157641,18.713237,-108.787800,638.895412,0.037752,11.019987,13.263181,-16.204007,1.0
107,69,1,10,-11,24.378943,-1.423613,-57.278292,638.651380,0.029134,16.812890,-1.067855,-17.571512,1.0
112,69,3,10,-11,78.605211,32.717337,-124.674506,638.971623,0.062979,6.554563,15.697553,-16.340517,1.0
88,69,1,23,-11,59.627702,8.857242,-95.143618,638.828547,0.000004,13.711768,9.782598,-16.868643,1.0
114,69,1,20,-11,60.364891,9.409642,-96.057101,638.833407,0.066774,13.478306,10.149418,-16.696679,1.0
127,69,1,20,-11,60.172088,9.257531,-95.817992,638.832263,0.109546,13.624169,10.308508,-16.558272,1.0
116,69,1,20,-11,59.988169,9.118368,-95.589474,638.831179,0.027860,13.652180,9.921548,-16.825549,1.0


In [75]:
tracks.sort_values('trackID')

,trackID,pdg,vx,vy,vz,vt,px,py,pz,eventID
17,63,-13,2.599931,5.780461,-1000.000000,68.166177,2.205410,0.708303,27.441719,1
16,69,-11,3.569605,4.482184,-34.896518,638.547219,15.004184,-7.480532,-17.746715,1
13,168,-13,5.000677,3.533811,-1000.000000,114.461760,0.353498,0.293159,27.486535,1
12,173,-11,5.796631,1.702258,-34.121101,723.091054,-25.740775,3.735758,-40.982284,1
14,404,-13,-0.061025,1.899658,-1000.000000,468.028921,-1.751591,0.512802,27.972889,1
11,407,-11,17.113040,-13.609033,-127.096666,633.852131,23.750739,22.599806,8.964159,1
10,411,22,19.121599,-10.938829,-126.211348,633.863712,7.260447,9.784165,2.577082,1
18,483,-11,-3.758463,7.317668,28.416645,513.852229,-36.132128,-2.351127,11.924348,1
15,651,-13,3.991748,0.159729,-1000.000000,438.674271,1.095563,0.320131,29.050429,1
19,658,-11,9.083620,5.158957,22.555047,559.595696,16.081014,-20.205157,24.086832,1


## `loading parquet`

In [28]:
hit_file_path = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_hits.parquet'
track_file_path = '/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/parquet/test_fast/all_tracks.parquet'

#### `Mu3eDataset.load_event`

In [29]:
# load events
all_hits = pd.read_parquet(hit_file_path)
all_tracks = pd.read_parquet(track_file_path)

hits = all_hits[all_hits['eventID'] == 0].reset_index(drop=True)
tracks = all_tracks[all_tracks['eventID'] == 0].reset_index(drop=True)

In [37]:
# add extra hit fields (geometric features)
hits["r"] = np.sqrt(hits["x"] ** 2 + hits["y"] ** 2)
hits["s"] = np.sqrt(hits["x"] ** 2 + hits["y"] ** 2 + hits["z"] ** 2)
hits["lambda"] = np.arccos(hits["z"] / hits["s"])                       # use our variable lambda
hits["phi"] = np.arctan2(hits["y"], hits["x"])
hits["eta"] = -np.log(np.tan(hits["lambda"] / 2))
hits["u"] = hits["x"] / (hits["x"] ** 2 + hits["y"] ** 2)
hits["v"] = hits["y"] / (hits["x"] ** 2 + hits["y"] ** 2)

# add extra track fields (kinematic features)
tracks["p"] = np.sqrt(tracks["px"] ** 2 + tracks["py"] ** 2 + tracks["pz"] ** 2)
tracks["pt"] = np.sqrt(tracks["px"] ** 2 + tracks["py"] ** 2)
tracks["eta"] = np.arctanh(tracks["pz"] / tracks["p"])
tracks["lambda"] = np.arccos(tracks["pz"] / tracks["p"])
tracks["phi"] = np.arctan2(tracks["py"], tracks["px"])
tracks["coslambda"] = np.cos(tracks["lambda"])
tracks["sinlambda"] = np.sin(tracks["lambda"])
tracks["cosphi"] = np.cos(tracks["phi"])
tracks["sinphi"] = np.sin(tracks["phi"])

# mark which hits are on a valid / reconstructable particle, for the hit filter
hits["on_valid_particle"] = hits["trackID"].isin(tracks["trackID"])

In [40]:
print(tracks)

   trackID  pdg         vx         vy          vz           vt         px  \
0     5826  -11   8.519007  -3.321426   25.927015  6635.948151 -24.844975   
1     9295  -11 -13.739107 -18.561076 -127.417946  6545.926486 -44.870896   
2    10967  -11  10.323480  -2.017735   22.316980  6653.607039  34.062474   
3    12835  -11   2.117287   1.949757  -42.400394  6482.033525  16.519859   
4    13748  -11  -0.880508 -23.284927   62.269823  6604.712338 -10.181432   

          py         pz  eventID          p         pt       eta    lambda  \
0   2.206126 -17.411388        0  30.418682  24.942730 -0.651072  2.180215   
1   1.049277 -14.113890        0  47.049975  44.883162 -0.309494  1.875464   
2  -5.453171  28.971081        0  45.047894  34.496221  0.763472  0.872234   
3  19.945178  13.349096        0  29.136133  25.898183  0.494984  1.094869   
4   5.430248 -22.540270        0  25.322183  11.539027 -1.422596  2.668447   

        phi  coslambda  sinlambda    cosphi    sinphi  
0  3.053029 

#### `Mu3eDataset.__getitem__`

In [38]:
# prepare containers
inputs = {}
targets = {}

# Load the event
num_tracks = len(tracks)

In [ ]:
# Build the input hits
for feature, fields in self.inputs.items():
    feature_hits = hits

    # Valid mask is all True for the feature-specific subset
    inputs[f"{feature}_valid"] = torch.full((len(feature_hits),), True).unsqueeze(0)
    targets[f"{feature}_valid"] = inputs[f"{feature}_valid"]

    for field in fields:
        inputs[f"{feature}_{field}"] = torch.from_numpy(feature_hits[field].values).unsqueeze(0).half()